## Load Text Files

In [12]:
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,DataCollatorForLanguageModeling, Trainer, TrainingArguments
import torch
from pathlib import Path

data_files = {
    "train": "../data/train.txt",
    "validation": "../data/validation.txt",
}

raw_ds = load_dataset("text", data_files=data_files)
raw_ds

Generating train split: 21828 examples [00:00, 66623.49 examples/s]
Generating validation split: 5457 examples [00:00, 76076.05 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 21828
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 5457
    })
})

## Load tokenizer and model

In [13]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = model.config.eos_token_id

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7337.64it/s]


## Tokenization

In [14]:
block_size = 128  

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=block_size,
    )

tokenized_ds = raw_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

Map: 100%|██████████| 5457/5457 [00:00<00:00, 6127.60 examples/s]


## Training

In [15]:
# Training args
training_args = TrainingArguments(
    output_dir="../models/finetuned-gpt2",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_dir="../logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    fp16=False,
)
# Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
)

train_result = trainer.train()
trainer.save_model("../models/finetuned-gpt2")
tokenizer.save_pretrained("../models/finetuned-gpt2")

train_result

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/Users/paripurna/Documents/Learning/gen_ai_learning/genai-learning/projects/project-4-fine-tuning-gen-model-quantization/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
500,3.587605,3.477592
1000,3.548766,3.430494
1500,3.386715,3.409097
2000,3.406524,3.395313
2500,3.360422,3.387557
2730,3.407456,3.385973


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.59it/s]
/Users/paripurna/Documents/Learning/gen_ai_learning/genai-learning/projects/project-4-fine-tuning-gen-model-quantization/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.53it/s]
/Users/paripurna/Documents/Learning/gen_ai_learning/genai-learning/projects/project-4-fine-tuning-gen-model-quantization/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.22it/s]
/Users/paripurna/Documents/Learning/gen_ai_learning/genai-learning/projects/project-4-fine-tuning-gen-model-quantization/venv/li

TrainOutput(global_step=2730, training_loss=3.4947610959901914, metrics={'train_runtime': 5744.4057, 'train_samples_per_second': 7.6, 'train_steps_per_second': 0.475, 'total_flos': 2851464635136000.0, 'train_loss': 3.4947610959901914, 'epoch': 2.0})

## Evaluate and generate from fine-tuned model

In [16]:

import numpy as np
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# reload fine-tuned model to verify it saves/loads
ft_model_path = "../models/finetuned-gpt2"
ft_tokenizer = AutoTokenizer.from_pretrained(ft_model_path)
ft_model = AutoModelForCausalLM.from_pretrained(ft_model_path).to(device)
ft_model.eval()

prompts = [
    "A game about space exploration",
    "A game with multiplayer action",
    "A story-driven adventure game",
    "A puzzle game for kids",
    "A horror survival game",
]

gen_kwargs = {
    "max_new_tokens": 80,
    "temperature": 0.8,
    "top_k": 50,
    "top_p": 0.95,
    "do_sample": True,
    "num_return_sequences": 1,
    "pad_token_id": ft_tokenizer.eos_token_id,
}

ft_outputs = []
latencies = []

for i, prompt in enumerate(prompts, 1):
    inputs = ft_tokenizer(prompt, return_tensors="pt").to(device)

    start = time.perf_counter()
    output_ids = ft_model.generate(**inputs, **gen_kwargs)
    end = time.perf_counter()

    text = ft_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    latencies.append(end - start)
    ft_outputs.append({"prompt": prompt, "output": text})

    print(f"\n--- Fine-tuned Output {i} ---")
    print("Prompt:", prompt)
    print("Generated:", text)

avg_ft_latency = float(np.mean(latencies))
print(f"\nAverage fine-tuned generation time: {avg_ft_latency:.4f} sec")

Device: mps


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 14296.57it/s]



--- Fine-tuned Output 1 ---
Prompt: A game about space exploration
Generated: A game about space exploration. The game contains a vast amount of levels, some are challenging and some are a bit easier. The game will give you a chance to see how to play. The game will have a lot of weapons, some are simple but some are very complex and complicated. In the game there are two weapons, a rocket launcher and a space elevator. Each weapon is equipped with a shield or a shield-

--- Fine-tuned Output 2 ---
Prompt: A game with multiplayer action
Generated: A game with multiplayer action in which you must control the ship using your joystick and the other players on the ship to control the ship. In this game you control the pilot of a ship using his or her own ship, your ship's abilities. As you use different ship types to achieve different goals, different challenges will be created. Game Features: - 4 different ship types - Different stages with different enemies - 3 difficulty levels -

--- 

## Save output and metrics from the fine-tuned

In [17]:
import pandas as pd
from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

# Save outputs
out_txt = results_dir / "finetuned_outputs.txt"
with out_txt.open("w", encoding="utf-8") as f:
    for i, item in enumerate(ft_outputs, 1):
        f.write(f"--- Fine-tuned Output {i} ---\n")
        f.write(f"Prompt: {item['prompt']}\n")
        f.write(f"Generated: {item['output']}\n\n")

# Save metrics
num_params = sum(p.numel() for p in ft_model.parameters())
model_size_mb = num_params * 4 / (1024 ** 2)

ft_row = {
    "model": "gpt2-finetuned",
    "avg_generation_time_sec": avg_ft_latency,
    "max_new_tokens": gen_kwargs["max_new_tokens"],
    "num_return_sequences": gen_kwargs["num_return_sequences"],
    "approx_model_size_mb": model_size_mb,
}

metrics_path = results_dir / "metrics.csv"
if metrics_path.exists():
    old = pd.read_csv(metrics_path)
    metrics_df = pd.concat([old, pd.DataFrame([ft_row])], ignore_index=True)
else:
    metrics_df = pd.DataFrame([ft_row])

metrics_df.to_csv(metrics_path, index=False)

print(f"Saved outputs to: {out_txt}")
print(f"Saved metrics to: {metrics_path}")
print(ft_row)

Saved outputs to: ../results/finetuned_outputs.txt
Saved metrics to: ../results/metrics.csv
{'model': 'gpt2-finetuned', 'avg_generation_time_sec': 1.3503245332001825, 'max_new_tokens': 80, 'num_return_sequences': 1, 'approx_model_size_mb': 474.7001953125}
